## A Minimal Example of MIQP

In [ ]:
%load_ext autoreload
import numpy as np
from phisolve.solvers.phi_miqp import PhiMIQPParams, PhiMIQP
from phisolve.backends.commons import BackendParams
from phisolve.refiners.pdqp import PDQP
from phisolve.refiners.pdqp_adhoc import PDQP as PDQPAdhoc
from phisolve.problems.miqp import MIQP
import scipy

In [ ]:
seed = 42

np.random.seed(seed)

n = 30
m = 50
k = 5
nbin = int(n / 3)
U = scipy.stats.ortho_group(n, seed).rvs()
eig = np.random.normal(-10, 5, (n))
Q = U.T @ np.diag(eig) @ U
w = np.random.normal(0, 10, (n))
x = np.random.uniform(0, 5, (n))
x[:nbin] = np.random.randint(0, 2, (nbin))
lbs = np.concatenate((np.zeros(nbin), np.min(x[nbin:]) * np.ones(n - nbin)))
ubs = np.concatenate((np.ones(nbin), np.max(x[nbin:]) * np.ones(n - nbin)))
A = np.random.normal(0, 1, (m, n))
b = np.random.normal(0, np.sqrt(n), (m))
infeas = A @ x > b
A[infeas] *= -1
b[infeas] *= -1
C = np.random.uniform(0, 1, (k, n))
d = C @ x

miqp = MIQP(Q, w, A=A, b=b, C=C, d=d, n_binary_vars=nbin, bounds=(lbs, ubs))

In [ ]:
n_shots = 100
n_steps = 10000
seed = 42
device = "cpu"

In [ ]:
import jax
from phisolve.utils.jax_utils import jax_device
jax.config.update("jax_platforms", jax_device(device))

In [ ]:
backend_params = BackendParams(n_shots=n_shots, n_steps=n_steps, seed=seed, device=device, lc_pr=10, slow_a=False)
refiner = PDQP(device=device, iterations=10000).refine
solver_params = PhiMIQPParams(refine=refiner, backend_params=backend_params)

solver = PhiMIQP(miqp)
res = solver.run(solver_params)

xs, cnts = res.refined_samples, res.sample_counts

objs = np.array([miqp.obj(x) for x in xs])
maxvios = np.array([miqp.max_vios(x) for x in xs])
feas = maxvios < 1e-4
minima = np.min(objs[feas])
minimizer = np.argmin(objs[feas])
succ = objs[feas] <= minima + 1e-4
minimizer_vios = maxvios[feas][minimizer]
succ_prob = np.sum(cnts[feas][succ]) / n_shots
print(minima, minimizer_vios, succ_prob)
assert minima <= -1075